# KvForge: Layer-Discriminative Bit Allocation


In [ ]:
import json, math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.cache_utils import DynamicCache

device = "cpu"
print("Device:", device)

# ===== KvForge Core =====

class LoRAConv1D(nn.Module):
    def __init__(self, orig, r=8, alpha=16.0):
        super().__init__()
        self.orig = orig; self.scaling = alpha / r
        in_f = orig.weight.shape[0]; out_f = orig.nf
        self.lora_A = nn.Parameter(torch.randn(in_f, r) * 0.02)
        self.lora_B = nn.Parameter(torch.zeros(r, out_f))
        self.active = True
    def activate(self, a=True): self.active = a
    def forward(self, x):
        h = self.orig(x)
        if self.active: h = h + (x @ self.lora_A @ self.lora_B) * self.scaling
        return h

def inject_lora(model, r=8, alpha=16.0):
    count = 0
    for n, m in model.named_modules():
        if n.endswith(".attn.c_attn") or n.endswith(".attn.c_proj"):
            parent = model; parts = n.split("."); child = parts[-1]
            for p in parts[:-1]:
                if p: parent = getattr(parent, p)
            setattr(parent, child, LoRAConv1D(m, r=r, alpha=alpha))
            count += 1
    print("  LoRA injected:", count, "modules")
    return count

def set_lora(m, a):
    for mod in m.modules():
        if hasattr(mod, "activate"): mod.activate(a)

def cache_mb(past):
    total = 0
    for layer in past:
        k, v = layer[0], layer[1]
        total += k.numel() * k.element_size() + v.numel() * v.element_size()
    return total / (1024**2)

# ---- Layer-discriminative bit allocation ----

def get_layer_bits(li, n_layers, scheme, target_bits):
    """Return bit width for layer li based on allocation scheme."""
    if scheme == "uniform":
        return target_bits
    ratio = li / max(n_layers - 1, 1)
    if scheme == "linear_increase":
        # Early layers = fewer bits, late layers = more bits
        min_b, max_b = 2, min(8, target_bits * 2)
        return max(min_b, int(min_b + (max_b - min_b) * ratio))
    elif scheme == "linear_decrease":
        # Early layers = more bits, late layers = fewer bits
        min_b, max_b = 2, min(8, target_bits * 2)
        return max(min_b, int(max_b - (max_b - min_b) * ratio))
    elif scheme == "extreme_decrease":
        # Early preserve, late aggressive quant
        if li < 0.25 * n_layers: return min(8, target_bits * 2)
        elif li < 0.5 * n_layers: return target_bits
        elif li < 0.75 * n_layers: return max(2, target_bits // 2)
        else: return 2
    elif scheme == "extreme_increase":
        # Early aggressive quant, late preserve
        if li < 0.25 * n_layers: return 2
        elif li < 0.5 * n_layers: return max(2, target_bits // 2)
        elif li < 0.75 * n_layers: return target_bits
        else: return min(8, target_bits * 2)
    return target_bits

def compress_layerwise(past, scheme, target_bits):
    """Compress KV cache with per-layer bit allocation."""
    n = len(past)
    dc = DynamicCache()
    for li, layer in enumerate(past):
        bits = get_layer_bits(li, n, scheme, target_bits)
        if bits >= 16:
            dc.update(layer[0], layer[1], layer[0].size(2))
            continue
        k, v = layer[0], layer[1]
        # Quantize K (uniform)
        mnk, mxk = k.min(-1, True).values, k.max(-1, True).values
        sk = (mxk - mnk).clamp(1e-8) / (2**bits - 1)
        dk = (((k - mnk) / sk).round().clamp(0, 2**bits-1).float() * sk + mnk).to(k.dtype)
        # Quantize V
        mnv, mxv = v.min(-1, True).values, v.max(-1, True).values
        sv = (mxv - mnv).clamp(1e-8) / (2**bits - 1)
        dv = (((v - mnv) / sv).round().clamp(0, 2**bits-1).float() * sv + mnv).to(v.dtype)
        dc.update(dk, dv, dk.size(2))
    return dc

def cache_mb_layerwise(past, scheme, target_bits):
    """Calculate cache size in MB considering per-layer bit widths."""
    n = len(past)
    total = 0
    for li, layer in enumerate(past):
        bits = get_layer_bits(li, n, scheme, target_bits)
        k, v = layer[0], layer[1]
        total += k.numel() * (bits / 8) + v.numel() * (bits / 8)
    return total / (1024**2)

def train_lora(model, texts, steps=60, lr=3e-3):
    tok = AutoTokenizer.from_pretrained("gpt2")
    tok.pad_token = tok.eos_token
    params = [p for n,p in model.named_parameters() if "lora" in n]
    opt = torch.optim.AdamW(params, lr=lr)
    model.train()
    losses = []
    for s in range(steps):
        text = texts[s % len(texts)]
        inp = tok(text, return_tensors="pt", truncation=True, max_length=128).to(device)
        ids = inp["input_ids"]
        out = model(ids)
        loss = F.cross_entropy(out.logits[0, :-1], ids[0, 1:])
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if s % 30 == 0:
            print("  Step %d | Loss: %.4f" % (s, loss.item()))
    model.eval()
    return losses

# ===== MAIN =====
print("=" * 70)
print("KvForge: Layer-Discriminative Bit Allocation")
print("=" * 70)

print("[1/3] Loading + training GPT-2 Small...", end=" ")
bm = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
inject_lora(bm, r=8)
texts = [
    "The transformer uses self-attention to process sequences in parallel.",
    "KV cache compression reduces memory by quantizing key-value pairs.",
    "LoRA adapters learn low-rank updates to attention projections.",
]
losses = train_lora(bm, texts, steps=60)
print("Loss: %.4f -> %.4f" % (losses[0], losses[-1]))

print("[2/3] Benchmarking layer-wise schemes...")
tok = AutoTokenizer.from_pretrained("gpt2")
tok.pad_token = tok.eos_token

schemes = ["uniform", "linear_increase", "linear_decrease", "extreme_decrease", "extreme_increase"]
target_bits = 4

prompt = "The transformer architecture revolutionized machine learning by introducing attention."
inp = tok(prompt, return_tensors="pt", truncation=True, max_length=96).to(device)
inp_ids = inp["input_ids"]

# Prefill once (Base Encode)
set_lora(bm, False)
with torch.no_grad():
    out = bm.generate(**inp, max_new_tokens=1, use_cache=True,
        pad_token_id=tok.eos_token_id, do_sample=False,
        return_dict_in_generate=True)
past = out.past_key_values
last_tok = out.sequences[:, -1:]
n_layers = len(past)
print("  Model has %d layers, FP16 cache: %.4f MB" % (n_layers, cache_mb(past)))

results = []
print()
print("  %-20s %6s %6s %8s %8s  LayerBits" % ("Scheme", "Prefill", "Decode", "Cache", "PPL"))
print("  " + "-"*80)

for scheme in schemes:
    layer_bits = [get_layer_bits(li, n_layers, scheme, target_bits) for li in range(n_layers)]
    past_c = compress_layerwise(past, scheme, target_bits)
    cm = cache_mb_layerwise(past, scheme, target_bits)

    # Decode with LoRA
    set_lora(bm, True)
    t0 = time.time()
    lt = last_tok.clone()
    with torch.no_grad():
        for _ in range(12):
            od = bm(lt, past_key_values=past_c, use_cache=True)
            past_c = od.past_key_values
            lt = od.logits[:, -1:].argmax(dim=-1)
    td = (time.time() - t0) * 1000
    set_lora(bm, False)

    # PPL
    with torch.no_grad():
        ob = bm(inp_ids)
    ppl = math.exp(F.cross_entropy(ob.logits[0, :-1], inp_ids[0, 1:]).item())

    bits_str = " ".join(str(b) for b in layer_bits[:3]) + "..."
    bits_str += " " + " ".join(str(b) for b in layer_bits[-3:])
    print("  %-20s %6.1f %6.1f %8.4f %8.2f  %s" % (scheme, 0, td, cm, ppl, bits_str))
    results.append({"scheme": scheme, "target_bits": target_bits,
                    "layer_bits": layer_bits, "decode_ms": round(td,2),
                    "cache_mb": round(cm,4), "perplexity": round(ppl,4)})

# Summary vs uniform
bl_c = results[0]["cache_mb"]
bl_p = results[0]["perplexity"]

print()
print("=" * 70)
print("SUMMARY vs uniform 4-bit (baseline)")
print("=" * 70)
print()
print("  %-20s %8s %8s %8s %8s" % ("Scheme", "Cache", "Savings", "PPL", "PPL diff"))
print("  " + "-"*52)
for r in results:
    cr = bl_c / r["cache_mb"] if r["cache_mb"] > 0 else 1
    pd = r["perplexity"] - bl_p
    vs_fp16 = cache_mb(past) / r["cache_mb"] if r["cache_mb"] > 0 else 1
    tag = "BEST" if r["perplexity"] <= bl_p else ("OK" if pd < 0.5 else "WARN")
    print("  %-20s %8.4f %7.1fx %8.2f %+8.4f [%s]" % (r["scheme"], r["cache_mb"], vs_fp16, r["perplexity"], pd, tag))

print()
print("  FP16 uniform: %.4f MB" % cache_mb(past))
best_scheme = min(results, key=lambda x: x["cache_mb"])
print("  Max savings: %.1fx (FP16 -> %s)" % (cache_mb(past) / best_scheme["cache_mb"], best_scheme["scheme"]))

with open("/kaggle/working/results.json", "w") as f:
    json.dump({"schemes": schemes, "target_bits": target_bits,
               "n_layers": n_layers, "results": results,
               "fp16_cache_mb": round(cache_mb(past),4)}, f, indent=2)
print()
print("results.json saved | Done!")
